In [1]:
import copy

import numpy as np
import pandas as pd

# ==== 1) 读取数据 ====
input_path = "fwwb01_realistic_with_anomalies_cleaned.csv"
output_path = "fwwb01_realistic_with_simulated_kpis.csv"

df = pd.read_csv(input_path)

# 统一类型
for c in ["hour", "dayofweek", "sqm", "airTemperature", "dewTemperature", "humidity"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# 若 hour/dayofweek 缺失，尝试从 timestamp 补齐
if ("timestamp" in df.columns) and (df["hour"].isna().any() or df["dayofweek"].isna().any()):
    ts = pd.to_datetime(df["timestamp"], errors="coerce")
    df.loc[df["hour"].isna(), "hour"] = ts.dt.hour
    # pandas: Monday=0, Sunday=6 -> 1~7
    df.loc[df["dayofweek"].isna(), "dayofweek"] = ts.dt.dayofweek + 1

# 读取后去重：避免重复 (building_id, timestamp) 导致后续 numpy 与 df 行数不一致
if "timestamp" in df.columns and "building_id" in df.columns:
    _n_dup = len(df)
    df = df.drop_duplicates(subset=["building_id", "timestamp"], keep="first").reset_index(drop=True)
    if len(df) < _n_dup:
        print(f"[dedupe] 读取后去重: 已删除 {_n_dup - len(df)} 行")

# 保证整型小时/星期（在去重之后，与 df 行对齐）
hour = df["hour"].fillna(0).astype(int).clip(0, 23)
dow = df["dayofweek"].fillna(1).astype(int).clip(1, 7)

# ==== 2) 三类建筑：办公楼 office / 教学楼 teaching / 图书馆 library ====
# 优先级：BUILDING_ID_TO_TYPE 覆盖 > building_type 列 > building_id 名称 > primaryspaceusage > 默认
BUILDING_ID_TO_TYPE = {
    # 例："某楼ID": "office",
    "Fox_public_Henry": "library",
}
# 无法识别时默认类型（校园数据多为教学楼，可按项目改为 "office" 或 "library"）
DEFAULT_BUILDING_TYPE = "teaching"

# primaryspaceusage 标准三类取值 → office / teaching / library（大小写不敏感）
PRIMARYSPACEUSAGE_TO_TYPE = {
    "office": "office",
    "education": "teaching",
    "public services": "library",
    "public service": "library",
    "public_services": "library",
}


def _norm_text(val):
    if val is None:
        return ""
    try:
        if isinstance(val, float) and np.isnan(val):
            return ""
    except TypeError:
        pass
    return str(val).strip().lower()


def classify_usage(row):
    bid = str(row.get("building_id", "")).strip()
    if bid in BUILDING_ID_TO_TYPE:
        return BUILDING_ID_TO_TYPE[bid]

    # 可选列：building_type，取值 office/teaching/library 或 办公/教学/图书 等
    bt = row.get("building_type", None)
    if bt is not None and (not pd.isna(bt)) and str(bt).strip() != "":
        bt_s = str(bt).strip().lower()
        if bt_s in ("office", "办公", "办公楼", "写字楼", "行政"):
            return "office"
        if bt_s in ("teaching", "教学", "教学楼", "教室", "教育"):
            return "teaching"
        if bt_s in ("library", "图书", "图书馆", "阅览"):
            return "library"

    # 你方数据 primaryspaceusage：Office / Education / Public services
    psu_key = str(row.get("primaryspaceusage", "")).strip().lower()
    if psu_key in PRIMARYSPACEUSAGE_TO_TYPE:
        return PRIMARYSPACEUSAGE_TO_TYPE[psu_key]

    bid_l = bid.lower()
    # building_id 字符串线索（中英文）
    if any(k in bid for k in ("办公", "写字楼", "行政")) or any(
        k in bid_l for k in ("office", "admin", "hq")
    ):
        return "office"
    if any(k in bid for k in ("教学", "教室")) or any(
        k in bid_l for k in ("teach", "classroom", "class_", "edu")
    ):
        return "teaching"
    if any(k in bid for k in ("图书", "阅览")) or "library" in bid_l:
        return "library"

    psu = _norm_text(row.get("primaryspaceusage", ""))
    if ("office" in psu) or ("办公" in psu):
        return "office"
    if (
        ("teaching" in psu)
        or ("class" in psu)
        or ("education" in psu)
        or ("教室" in psu)
        or ("教学" in psu)
    ):
        return "teaching"
    if ("library" in psu) or ("图书" in psu) or ("阅览" in psu):
        return "library"

    return DEFAULT_BUILDING_TYPE


usage = df.apply(classify_usage, axis=1)
# sim_building_type 不写入输出 CSV，仅内存中的 usage 供仿真使用
weekday = dow <= 5

# ==== 3) 时间系数 S(hour, dayofweek) ====
def S_office(h, is_weekday):
    if not is_weekday:
        return 0.08
    if 0 <= h <= 6:
        return 0.08
    if h == 7:
        return 0.2
    if h == 8:
        return 0.6
    if 9 <= h <= 17:
        return 1.0
    if 18 <= h <= 20:
        return 0.35
    return 0.10


def S_teaching(h, is_weekday):
    if not is_weekday:
        return 0.15
    if 0 <= h <= 6:
        return 0.08
    if h == 7:
        return 0.2
    if 8 <= h <= 12:
        return 1.0
    if h == 13:
        return 0.45
    if 14 <= h <= 17:
        return 1.0
    if 18 <= h <= 21:
        return 0.75
    return 0.10


def S_library(h, is_weekday):
    # 图书馆夜间仍保留少量运行
    if 0 <= h <= 6:
        return 0.10
    if h == 7:
        return 0.2
    if 8 <= h <= 21:
        return 0.85
    return 0.15


S = np.zeros(len(df), dtype=float)
for i, (u, h, wd) in enumerate(zip(usage.values, hour.values, weekday.values)):
    if u == "office":
        S[i] = S_office(h, wd)
    elif u == "teaching":
        S[i] = S_teaching(h, wd)
    else:
        S[i] = S_library(h, wd)

S = np.clip(S, 0.08, 1.0)
# sim_time_factor_s 不写入输出 CSV

# ==== 4) 国标参数模板化（请按项目条文值校准） ====
# 说明：
# - 此模板给出可运行的“参考默认值”，方便后续你替换为项目采用的国标/地标条文数值。
# - 参数按“建筑类型 + 季节”管理，仿真主逻辑不再写死。
# - season 取值: cooling/heating/transition
GB_PARAM_TEMPLATE = {
    "office": {
        "internal": {
            # 用水峰值 L/(m2·h)：三类收窄差距，避免同温同 sqm 下水耗差数量级
            "W_peak": 0.36,
            "P_base": 22.0,
            "q_person": 12.0,
            # 围护传热简化系数 W/(m2·K)；用水基线 L/(m2·h)；高温补水系数 L/(m2·h)/℃（Ta>26 部分）
            "U_envelope": 5.0,
            "water_base_wm2": 0.004,
            "water_weather_wm2_per_deg": 0.002,
        },
        "hvac": {
            "cop": {"cooling": 3.6, "heating": 2.7},
            "pump_wm2": {"run": 1.00, "standby": 0.15},
            "vent_wm2": {"active": 2.2, "minimum": 0.6},
            "hysteresis": {
                "cool_on_ta": 24.0,
                "cool_off_ta": 22.0,
                "cool_on_q": 22.0,
                "cool_off_q": 14.0,
                "heat_on_ta": 14.0,
                "heat_off_ta": 16.0,
                "heat_on_qenv": -18.0,
            },
            "extreme_force_on": {"low_ta": 8.0, "high_ta": 30.0},
            "min_on_load_kw": 0.05,
        },
    },
    "teaching": {
        "internal": {
            "W_peak": 0.48,
            "P_base": 24.0,
            "q_person": 13.0,
            "U_envelope": 5.3,
            "water_base_wm2": 0.008,
            "water_weather_wm2_per_deg": 0.0022,
        },
        "hvac": {
            "cop": {"cooling": 3.4, "heating": 2.6},
            "pump_wm2": {"run": 1.05, "standby": 0.16},
            "vent_wm2": {"active": 2.6, "minimum": 0.7},
            "hysteresis": {
                "cool_on_ta": 24.0,
                "cool_off_ta": 22.0,
                "cool_on_q": 24.0,
                "cool_off_q": 15.0,
                "heat_on_ta": 14.0,
                "heat_off_ta": 16.0,
                "heat_on_qenv": -20.0,
            },
            "extreme_force_on": {"low_ta": 7.0, "high_ta": 31.0},
            "min_on_load_kw": 0.05,
        },
    },
    "library": {
        "internal": {
            "W_peak": 0.30,
            "P_base": 20.0,
            "q_person": 11.0,
            "U_envelope": 4.6,
            "water_base_wm2": 0.0028,
            "water_weather_wm2_per_deg": 0.0015,
        },
        "hvac": {
            "cop": {"cooling": 3.8, "heating": 2.9},
            "pump_wm2": {"run": 0.85, "standby": 0.12},
            "vent_wm2": {"active": 1.8, "minimum": 0.5},
            "hysteresis": {
                "cool_on_ta": 24.5,
                "cool_off_ta": 22.5,
                "cool_on_q": 20.0,
                "cool_off_q": 13.0,
                "heat_on_ta": 13.5,
                "heat_off_ta": 16.0,
                "heat_on_qenv": -16.0,
            },
            "extreme_force_on": {"low_ta": 7.0, "high_ta": 31.0},
            "min_on_load_kw": 0.04,
        },
    },
}

# ==== 4b) 同一校园、共用同一条室外气象时：统一围护与设备参数 ====
# True：三类建筑的 U、冷热源 COP、泵/新风、滞回、用水气象项与 office 一致，仅保留
#   - 不同的日程 S(h)
#   - 略有差异的内热 P_base/q_person、W_peak（幅度已收窄）
# 这样“同温度”下各楼单位面积负荷/能耗同量级；总能耗仍正比于各自 sqm。
# False：使用上面 GB_PARAM_TEMPLATE 中各类型独立参数（适合不同围护/不同机房效率的对比研究）。
UNIFY_CLIMATE_AND_PLANT_ACROSS_TYPES = True

if UNIFY_CLIMATE_AND_PLANT_ACROSS_TYPES:
    ref = GB_PARAM_TEMPLATE["office"]
    ref_hvac = copy.deepcopy(ref["hvac"])
    ref_u = ref["internal"]["U_envelope"]
    ref_wb = ref["internal"]["water_base_wm2"]
    ref_ww = ref["internal"]["water_weather_wm2_per_deg"]
    for u in GB_PARAM_TEMPLATE:
        GB_PARAM_TEMPLATE[u]["internal"]["U_envelope"] = ref_u
        GB_PARAM_TEMPLATE[u]["internal"]["water_base_wm2"] = ref_wb
        GB_PARAM_TEMPLATE[u]["internal"]["water_weather_wm2_per_deg"] = ref_ww
        GB_PARAM_TEMPLATE[u]["hvac"] = copy.deepcopy(ref_hvac)


def param_array(usage_series, extractor):
    return usage_series.map(lambda u: extractor(GB_PARAM_TEMPLATE[u])).to_numpy()


W_peak = param_array(usage, lambda x: x["internal"]["W_peak"])
P_base = param_array(usage, lambda x: x["internal"]["P_base"])
q_person = param_array(usage, lambda x: x["internal"]["q_person"])

cop_cooling = param_array(usage, lambda x: x["hvac"]["cop"]["cooling"])
cop_heating = param_array(usage, lambda x: x["hvac"]["cop"]["heating"])

pump_run_wm2 = param_array(usage, lambda x: x["hvac"]["pump_wm2"]["run"])
pump_standby_wm2 = param_array(usage, lambda x: x["hvac"]["pump_wm2"]["standby"])
vent_active_wm2 = param_array(usage, lambda x: x["hvac"]["vent_wm2"]["active"])
vent_min_wm2 = param_array(usage, lambda x: x["hvac"]["vent_wm2"]["minimum"])

cool_on_ta = param_array(usage, lambda x: x["hvac"]["hysteresis"]["cool_on_ta"])
cool_off_ta = param_array(usage, lambda x: x["hvac"]["hysteresis"]["cool_off_ta"])
cool_on_q = param_array(usage, lambda x: x["hvac"]["hysteresis"]["cool_on_q"])
cool_off_q = param_array(usage, lambda x: x["hvac"]["hysteresis"]["cool_off_q"])
heat_on_ta = param_array(usage, lambda x: x["hvac"]["hysteresis"]["heat_on_ta"])
heat_off_ta = param_array(usage, lambda x: x["hvac"]["hysteresis"]["heat_off_ta"])
heat_on_qenv = param_array(usage, lambda x: x["hvac"]["hysteresis"]["heat_on_qenv"])

force_on_low_ta = param_array(usage, lambda x: x["hvac"]["extreme_force_on"]["low_ta"])
force_on_high_ta = param_array(usage, lambda x: x["hvac"]["extreme_force_on"]["high_ta"])
min_on_load_kw = param_array(usage, lambda x: x["hvac"]["min_on_load_kw"])

U_env = param_array(usage, lambda x: x["internal"]["U_envelope"])
water_base_wm2 = param_array(usage, lambda x: x["internal"]["water_base_wm2"])
water_weather_wm2 = param_array(usage, lambda x: x["internal"]["water_weather_wm2_per_deg"])

sqm = df["sqm"].fillna(df["sqm"].median()).to_numpy()
Ta = df["airTemperature"].ffill().bfill().fillna(20.0).to_numpy()
Td = df["dewTemperature"].ffill().bfill().fillna(10.0).to_numpy()
RH_base = df["humidity"].ffill().bfill().fillna(50.0).to_numpy()

# ==== 5) season 判定（用于模板扩展） ====
if "timestamp" in df.columns:
    ts = pd.to_datetime(df["timestamp"], errors="coerce")
    month = ts.dt.month.fillna(0).astype(int).to_numpy()
else:
    month = np.zeros(len(df), dtype=int)

season = np.full(len(df), "transition", dtype=object)
season[np.isin(month, [6, 7, 8, 9])] = "cooling"
season[np.isin(month, [11, 12, 1, 2, 3])] = "heating"
if (month == 0).all():
    season[Ta >= 26.0] = "cooling"
    season[Ta <= 8.0] = "heating"

# ==== 6) 用水仿真（增加基线与天气敏感性），单位：升 L ====
# W_peak / water_base_wm2 / water_weather 均为 L/(m2·h)；与 CSV 每行时间步一致时，本列为该步总用水量（L），
# 若时间步为 1 小时则数值等价于 L/h。
rng = np.random.default_rng(20260320)

water_active = sqm * W_peak * S
water_base = sqm * water_base_wm2
water_weather = np.maximum(Ta - 26.0, 0.0) * sqm * water_weather_wm2

water_ideal = water_active + water_base + water_weather
water_noise = rng.normal(0.0, 0.06 * np.maximum(water_ideal, 1e-6), size=len(df))
water_l = np.maximum(0.0, water_ideal + water_noise)
water_l_nominal = np.maximum(0.0, water_ideal)

# ==== 7) Q_load 与工况判定（按类型参数化滞回 + 围护 U） ====
T_in = np.where(Ta < 18.0, 20.0, 26.0)

q_env = U_env * (Ta - T_in)
q_internal = (P_base * S) + (q_person * S)
q_latent = np.maximum(0.0, 15.0 * (Td - 15.0)) * S
q_total = q_env + q_internal + q_latent

cool_demand = (Ta >= cool_on_ta) | (q_total >= cool_on_q)
heat_demand = (Ta <= heat_on_ta) | ((q_env <= heat_on_qenv) & (Ta < (heat_off_ta + 1.0)))

is_cooling = np.zeros(len(df), dtype=bool)
is_heating = np.zeros(len(df), dtype=bool)

if "building_id" in df.columns:
    group_key = df["building_id"].fillna("unknown").astype(str)
else:
    group_key = pd.Series(["all"] * len(df), index=df.index)

if "timestamp" in df.columns:
    ts_sort = pd.to_datetime(df["timestamp"], errors="coerce")
else:
    ts_sort = pd.Series(np.arange(len(df)), index=df.index)

for bid in group_key.unique():
    idx = np.where(group_key.values == bid)[0]
    ord_idx = idx[np.argsort(ts_sort.iloc[idx].values)]

    mode = 0  # 0=过渡, 1=制冷, -1=供暖
    for j in ord_idx:
        if mode == 1 and (Ta[j] <= cool_off_ta[j]) and (q_total[j] <= cool_off_q[j]):
            mode = 0
        elif mode == -1 and (Ta[j] >= heat_off_ta[j]):
            mode = 0

        if mode == 0:
            if heat_demand[j]:
                mode = -1
            elif cool_demand[j]:
                mode = 1

        is_cooling[j] = mode == 1
        is_heating[j] = mode == -1

is_transition = ~(is_cooling | is_heating)

Q_load = np.zeros(len(df), dtype=float)
Q_load[is_cooling] = sqm[is_cooling] * np.maximum(
    0.0,
    q_env[is_cooling] + q_internal[is_cooling] + q_latent[is_cooling],
) / 1000.0
Q_load[is_heating] = sqm[is_heating] * np.maximum(
    0.0,
    np.abs(q_env[is_heating]) - q_internal[is_heating],
) / 1000.0

hvac_on = (Q_load > min_on_load_kw) & ((S > 0.08) | (Ta < force_on_low_ta) | (Ta > force_on_high_ta))
Q_load = np.where(hvac_on, Q_load, 0.0)

Q_design = sqm * 100.0 / 1000.0
PLR = np.clip(np.divide(Q_load, np.maximum(Q_design, 1e-6)), 0.0, 1.0)

# ==== 8) 空调供回水温度（额定无噪声 + 实际含噪声） ====
sim_supply_nominal = Ta.astype(float).copy()
sim_return_nominal = Ta.astype(float).copy()
idx = is_cooling & hvac_on
sim_supply_nominal[idx] = 7.0
sim_return_nominal[idx] = 7.0 + 5.0 * PLR[idx]
idx = is_heating & hvac_on
sim_supply_nominal[idx] = 60.0
sim_return_nominal[idx] = 60.0 - 10.0 * PLR[idx]

sim_supply_temp_c = sim_supply_nominal.copy()
sim_return_temp_c = sim_return_nominal.copy()
idx = is_cooling & hvac_on
sim_supply_temp_c[idx] = sim_supply_nominal[idx] + rng.normal(0.0, 0.2, size=idx.sum())
sim_return_temp_c[idx] = sim_return_nominal[idx] + rng.normal(0.0, 0.2, size=idx.sum())
idx = is_heating & hvac_on
sim_supply_temp_c[idx] = sim_supply_nominal[idx] + rng.normal(0.0, 0.5, size=idx.sum())
sim_return_temp_c[idx] = sim_return_nominal[idx] + rng.normal(0.0, 0.5, size=idx.sum())

# ==== 9) hac_energy（COP、泵耗、新风全部参数化） ====
# --- 机理可解释：有效 COP 短时波动（不对 hac 再乘“黑箱”系数）---
# 乘子 ~ LogNormal，均值≈1：对应部分负荷效率偏离、室外工况、换热器脏堵、控制滞后等导致的
# 瞬时有效 COP 相对名义值 COP 的波动；仅作用于 Q_load/COP 项，泵与新风仍用确定性公式。
COP_EFFECTIVE_STOCHASTIC_ENABLED = True
COP_EFFECTIVE_LOGSIGMA_COOLING = 0.045
COP_EFFECTIVE_LOGSIGMA_HEATING = 0.055
COP_EFFECTIVE_CLIP_COOL = (2.3, 6.0)
COP_EFFECTIVE_CLIP_HEAT = (1.9, 4.2)

E_pump_run = sqm * pump_run_wm2 / 1000.0
E_pump_standby = sqm * pump_standby_wm2 / 1000.0
E_ventilation_active = np.where(S > 0.12, sqm * vent_active_wm2 / 1000.0, sqm * vent_min_wm2 / 1000.0)

n_h = len(df)
if COP_EFFECTIVE_STOCHASTIC_ENABLED:
    mult_c = np.exp(
        rng.normal(-0.5 * COP_EFFECTIVE_LOGSIGMA_COOLING**2, COP_EFFECTIVE_LOGSIGMA_COOLING, size=n_h)
    )
    mult_h = np.exp(
        rng.normal(-0.5 * COP_EFFECTIVE_LOGSIGMA_HEATING**2, COP_EFFECTIVE_LOGSIGMA_HEATING, size=n_h)
    )
    cop_eff_cooling = np.clip(
        cop_cooling * mult_c, COP_EFFECTIVE_CLIP_COOL[0], COP_EFFECTIVE_CLIP_COOL[1]
    )
    cop_eff_heating = np.clip(
        cop_heating * mult_h, COP_EFFECTIVE_CLIP_HEAT[0], COP_EFFECTIVE_CLIP_HEAT[1]
    )
else:
    cop_eff_cooling = np.asarray(cop_cooling, dtype=float).copy()
    cop_eff_heating = np.asarray(cop_heating, dtype=float).copy()

hac_energy = np.zeros(len(df), dtype=float)

cool_idx = is_cooling & hvac_on
heat_idx = is_heating & hvac_on
tran_idx = ~(cool_idx | heat_idx)

hac_energy[cool_idx] = Q_load[cool_idx] / np.maximum(cop_eff_cooling[cool_idx], 1e-6) + E_pump_run[cool_idx] + 0.5 * E_ventilation_active[cool_idx]
hac_energy[heat_idx] = Q_load[heat_idx] / np.maximum(cop_eff_heating[heat_idx], 1e-6) + E_pump_run[heat_idx] + 0.4 * E_ventilation_active[heat_idx]
hac_energy[tran_idx] = E_pump_standby[tran_idx] + E_ventilation_active[tran_idx]

hac_energy_nominal = np.zeros(len(df), dtype=float)
hac_energy_nominal[cool_idx] = Q_load[cool_idx] / np.maximum(cop_cooling[cool_idx], 1e-6) + E_pump_run[cool_idx] + 0.5 * E_ventilation_active[cool_idx]
hac_energy_nominal[heat_idx] = Q_load[heat_idx] / np.maximum(cop_heating[heat_idx], 1e-6) + E_pump_run[heat_idx] + 0.4 * E_ventilation_active[heat_idx]
hac_energy_nominal[tran_idx] = E_pump_standby[tran_idx] + E_ventilation_active[tran_idx]

sim_cop_effective_used = np.full(n_h, np.nan, dtype=float)
sim_cop_effective_used[cool_idx] = cop_eff_cooling[cool_idx]
sim_cop_effective_used[heat_idx] = cop_eff_heating[heat_idx]

# ==== 10) electricity_kwh ====
E_base = (sqm * P_base * S) / 1000.0
# 空调侧随机已由「有效 COP」机理项体现；此处噪声仅表示照明/插座等非空调分项的表计与模型不确定，避免对 hac 二次放大。
elec_noise = rng.normal(0.0, 0.04 * np.maximum(E_base, 1e-6), size=len(df))
electricity_kwh_nominal = np.maximum(0.0, E_base + hac_energy_nominal)
electricity_kwh = np.maximum(0.0, E_base + hac_energy + elec_noise)

# ==== 11) humidity_rh_pct ====
humidity_rh_pct = RH_base.copy().astype(float)

idx = is_cooling & hvac_on
humidity_rh_pct[idx] = humidity_rh_pct[idx] - (7.0 + 11.0 * PLR[idx]) * np.maximum(S[idx], 0.12)

idx = is_heating & hvac_on
humidity_rh_pct[idx] = humidity_rh_pct[idx] - (3.0 + 5.0 * PLR[idx]) * np.maximum(S[idx], 0.12)

off_idx = ~hvac_on
humidity_rh_pct[off_idx] = 0.9 * humidity_rh_pct[off_idx] + 0.1 * RH_base[off_idx]

humidity_rh_pct_nominal = np.clip(humidity_rh_pct.copy(), 20.0, 95.0)
humidity_rh_pct += rng.normal(0.0, 1.2, size=len(df))
humidity_rh_pct = np.clip(humidity_rh_pct, 20.0, 95.0)

# ==== 12) 写回结果 ====
# 以下列不输出到 CSV：hour, dayofweek, sim_building_type, sim_time_factor_s, sim_season,
# sim_cop_effective_used, electricity_kwh_per_sqm, hac_energy_kwh_per_sqm
df["water_l"] = water_l.round(2)
df["electricity_kwh"] = electricity_kwh.round(4)
df["sim_supply_temp_c"] = sim_supply_temp_c.round(3)
df["sim_return_temp_c"] = sim_return_temp_c.round(3)
df["hac_energy"] = hac_energy.round(4)
df["humidity_rh_pct"] = humidity_rh_pct.round(2)
# 不含随机噪声的额定值（与 inject/detect 基线一致）
df["water_l_nominal"] = water_l_nominal.round(2)
df["electricity_kwh_nominal"] = electricity_kwh_nominal.round(4)
df["sim_supply_temp_c_nominal"] = sim_supply_nominal.round(3)
df["sim_return_temp_c_nominal"] = sim_return_nominal.round(3)
df["hac_energy_nominal"] = hac_energy_nominal.round(4)
df["humidity_rh_pct_nominal"] = humidity_rh_pct_nominal.round(2)

_cols_no_export = [
    "hour",
    "dayofweek",
    "sim_building_type",
    "sim_time_factor_s",
    "sim_season",
    "sim_cop_effective_used",
    "electricity_kwh_per_sqm",
    "hac_energy_kwh_per_sqm",
]
df_out = df.drop(columns=[c for c in _cols_no_export if c in df.columns], errors="ignore")
df_out.to_csv(output_path, index=False, encoding="utf-8-sig")

print("Done. output:", output_path)
print(
    df_out[
        [
            "timestamp",
            "building_id",
            "primaryspaceusage",
            "water_l",
            "electricity_kwh",
            "sim_supply_temp_c",
            "sim_return_temp_c",
            "hac_energy",
            "humidity_rh_pct",
        ]
    ].head(10)
)
print("内部业态 usage counts:\n", usage.value_counts())
if "building_id" in df.columns:
    _sqm_series = df["sqm"].fillna(df["sqm"].median())
    _dbg = pd.DataFrame(
        {
            "building_id": df["building_id"].values,
            "sim_building_type": usage.values,
            "sqm": _sqm_series.values,
            "electricity_kwh": electricity_kwh,
        }
    )
    _dbg["_inten"] = _dbg["electricity_kwh"] / np.maximum(_dbg["sqm"], 1.0)
    _g = _dbg.groupby(["building_id", "sim_building_type"], dropna=False).agg(
        sqm_med=("sqm", "median"),
        elec_kwh_mean=("electricity_kwh", "mean"),
        inten_mean=("_inten", "mean"),
    ).reset_index()
    print("各楼：median sqm、小时均电耗、单位面积强度(诊断用，不写入 CSV):\n", _g.to_string(index=False))


[dedupe] 读取后去重: 已删除 3 行
Done. output: fwwb01_realistic_with_simulated_kpis.csv
             timestamp       building_id primaryspaceusage  water_l  \
0  2016-01-02 00:00:00  Fox_public_Henry   Public services    40.08   
1  2016-01-02 01:00:00  Fox_public_Henry   Public services    38.61   
2  2016-01-02 02:00:00  Fox_public_Henry   Public services    44.72   
3  2016-01-02 03:00:00  Fox_public_Henry   Public services    45.34   
4  2016-01-02 04:00:00  Fox_public_Henry   Public services    38.89   
5  2016-01-02 05:00:00  Fox_public_Henry   Public services    40.94   
6  2016-01-02 06:00:00  Fox_public_Henry   Public services    42.22   
7  2016-01-02 07:00:00  Fox_public_Henry   Public services    76.46   
8  2016-01-02 08:00:00  Fox_public_Henry   Public services   310.14   
9  2016-01-02 09:00:00  Fox_public_Henry   Public services   358.79   

   electricity_kwh  sim_supply_temp_c  sim_return_temp_c  hac_energy  \
0          27.0498             60.200             55.406     24.343